In [53]:
import qiime2 as q2
from qiime2 import Artifact, Metadata
import pandas as pd
from biom import Table
from biom.util import biom_open
from qiime2.plugins.feature_table.methods import filter_samples
from collections import defaultdict, Counter


In [54]:
def truncate_to_genus(taxonomy_string):
    if pd.isna(taxonomy_string):
        return taxonomy_string
    parts = taxonomy_string.split('; ')
    truncated_parts = [part for part in parts if not part.startswith('s__')]
    return '; '.join(truncated_parts)

# Load taxonomy and truncate to genus level
taxonomy_path = '../Metadata/174116_taxonomy.tsv'
taxonomy_df = pd.read_csv(taxonomy_path, sep='\t', index_col=0)
genus_taxonomy_map = {id: truncate_to_genus(taxon) for id, taxon in taxonomy_df['Taxon'].items()}

# Load metadata
metadata = Metadata.load('../Metadata/57610_57610_analysis_mapping.txt')

# Define paths and process in loop
table_paths = {
    'V1-V3': '../Data/16S/Tables/174950_rarefied_table_unannotated_absolute_abundances.qza',
    'V4': '../Data/16S/Tables/174951_rarefied_table_unannotated_absolute_abundances.qza'
}

for primer, path in table_paths.items():
    table = Artifact.load(path)
    
    # Filter samples using QIIME2 plugin method
    filtered = filter_samples(
        table=table,
        metadata=metadata,
        where='sample_type!="skin"',
        exclude_ids=True
    ).filtered_table

    # Convert to DataFrame
    df = filtered.view(pd.DataFrame)

    # Step 1: Map each ASV to its genus
    genus_names = [genus_taxonomy_map.get(idx, 'Unclassified') for idx in df.columns]

    # Step 2: Count how many times each genus appears
    genus_counts = Counter(genus_names)

    # Step 3: Build new column names with |ASV# if genus appears more than once
    seen = defaultdict(int)
    new_col_names = []
    for genus in genus_names:
        seen[genus] += 1
        if genus_counts[genus] > 1:
            new_col_names.append(f"{genus}|ASV{seen[genus]}")
        else:
            new_col_names.append(genus)

    # Step 4: Apply new names and transpose
    df_renamed = df.copy()
    df_renamed.columns = new_col_names
    df_renamed = df_renamed.T  # Transpose so features = rows

    print(df_renamed.index)

    # Convert to BIOM Table
    biom_table = Table(
        df_renamed.values,
        observation_ids=df_renamed.index.tolist(),
        sample_ids=df_renamed.columns.tolist()
    )

    # Save to .biom file
    output_fp = f'../Data/16S/Tables/16S_{primer}_Genus_uncollapsed_absolute.biom'
    with biom_open(output_fp, 'w') as f:
        biom_table.to_hdf5(f, f"{primer.upper()} genus-level filtered table")


Index(['d__Bacteria; p__Actinobacteriota; c__Actinobacteria; o__Corynebacteriales; f__Corynebacteriaceae; g__Corynebacterium|ASV1',
       'd__Bacteria; p__Proteobacteria; c__Alphaproteobacteria; o__Rhizobiales; f__Xanthobacteraceae|ASV1',
       'd__Bacteria; p__Proteobacteria; c__Gammaproteobacteria; o__Burkholderiales; f__Comamonadaceae|ASV1',
       'd__Bacteria; p__Bacteroidota; c__Bacteroidia; o__Bacteroidales; f__Porphyromonadaceae; g__Porphyromonas|ASV1',
       'd__Bacteria; p__Firmicutes; c__Clostridia; o__Oscillospirales; f__Ruminococcaceae; g__Subdoligranulum',
       'd__Bacteria; p__Proteobacteria; c__Gammaproteobacteria; o__Pseudomonadales; f__Moraxellaceae',
       'd__Bacteria; p__Proteobacteria; c__Gammaproteobacteria; o__Burkholderiales; f__Neisseriaceae; g__Neisseria|ASV1',
       'd__Bacteria; p__Proteobacteria; c__Gammaproteobacteria; o__Xanthomonadales; f__Xanthomonadaceae; g__Xanthomonas',
       'd__Bacteria; p__Proteobacteria; c__Gammaproteobacteria; o__Pseudo